# AegisDesk Kaggle DPO Notebook

This notebook is the Kaggle-safe fallback path for AegisDesk when GRPO keeps collapsing into identical reward groups.

It trains a small QLoRA DPO adapter on preference pairs, which is much more stable than online RL on a T4 when the policy keeps repeating the same safe action.

**Use this when:**
- the GRPO notebook keeps logging identical reward groups
- advantage stays near zero after the core-task probe
- you want a practical Kaggle path instead of more reward-shaping work


In [ ]:
%%capture
!pip install -U unsloth trl transformers datasets peft accelerate bitsandbytes sentencepiece protobuf

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

REPO_DIR = Path('/kaggle/working/AegisDesk')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', 'https://github.com/kumarabhik/AegisDesk.git', str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR), '--quiet'], check=True)
os.chdir(REPO_DIR)

HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN

MODEL_NAME = os.environ.get('AEGIS_DPO_MODEL_NAME', 'Qwen/Qwen2.5-1.5B-Instruct')
DATASET_PATH = 'training/data/support_pref.jsonl'
PROJECT_PREF_PATH = 'training/data/aegisdesk_pref.jsonl'
OUTPUT_DIR = '/kaggle/working/aegisdesk-dpo'

print('Repo:', REPO_DIR)
print('Model:', MODEL_NAME)
print('Dataset:', DATASET_PATH)
print('Project mix output:', PROJECT_PREF_PATH)
print('Output:', OUTPUT_DIR)


## Optional: Build a More AegisDesk-Focused Preference Set

If you already harvested wins and fails, switch `DATASET_PATH` to `training/data/aegisdesk_pref.jsonl` after this cell.

In [ ]:
build_cmd = [
    sys.executable,
    '-m', 'training.build_aegisdesk_preference_corpus',
    '--output', PROJECT_PREF_PATH,
    '--upsample-project-pairs', '4',
]
result = subprocess.run(build_cmd, text=True, capture_output=True)
print(result.stdout)
if result.returncode != 0:
    print('Project-specific preference build failed; falling back to the base preference dataset.')
    print(result.stderr)
else:
    DATASET_PATH = PROJECT_PREF_PATH
    print('Using project-mixed dataset:', DATASET_PATH)


In [ ]:
preview_cmd = [
    sys.executable,
    '-c',
    "import json; from itertools import islice; p='{}'; rows=[json.loads(x) for x in islice(open(p, encoding='utf-8'), 2)]; print(rows[0].keys()); print(rows[0]['prompt'][:300]); print(rows[0]['chosen'][:300])".format(DATASET_PATH),
]
subprocess.run(preview_cmd, check=True)

In [ ]:
train_cmd = [
    sys.executable,
    '-m', 'training.train_unsloth_dpo',
    '--dataset', DATASET_PATH,
    '--output', OUTPUT_DIR,
    '--model', MODEL_NAME,
    '--epochs', '1.0',
    '--per-device-train-batch-size', '1',
    '--gradient-accumulation-steps', '8',
    '--logging-steps', '5',
    '--save-steps', '50',
    '--min-score-gap', '0.05',
    '--report-to', 'none',
    '--run-name', 'aegisdesk-kaggle-dpo',
]
print('Running:', ' \n'.join(train_cmd[:3]), '...')
subprocess.run(train_cmd, check=True)


## Next Step

After training, evaluate the saved adapter with the project benchmark script or your preferred AegisDesk eval flow.

If you want to compare it directly against the GRPO run, keep the same benchmark seeds and core tasks first.
